In [1]:
# ==========================================================
# Interactive Photosynthesis Simulation Notebook
#
# This interactive simulation explores how environmental
# conditions influence plant growth and photosynthesis.
# Using Python, the model simulates how different plant
# types (C3, C4, and CAM) respond to variations in light
# intensity, temperature, and soil nutrients such as
# nitrogen, phosphorus, and potassium.
#
# The simulation models glucose production, biomass
# allocation, leaf area expansion, and plant health
# over time. Interactive sliders allow users to adjust
# environmental conditions and observe how plant growth
# responds dynamically.
#
# Author: Dr. Temesgen Deressa
# Date: March 5, 2026
# ==========================================================

import random
from enum import Enum
from typing import Dict
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, IntSlider
import numpy as np

# ------------------------------
# 1. Environment Setup
# ------------------------------

class LightIntensity(Enum):
    NONE = 0
    LOW = 1
    MEDIUM = 2
    HIGH = 3
    VERY_HIGH = 4

class Environment:
    def __init__(self, soil_nutrients: Dict[str, float], co2_ppm: float, temperature: float):
        self.soil_nutrients = soil_nutrients
        self.co2_ppm = co2_ppm
        self.temperature = temperature
        self.light = LightIntensity.MEDIUM

    def update_conditions(self, light_val, temp_val, nutrients_val):
        self.light = LightIntensity(light_val)
        self.temperature = temp_val
        for nutrient in self.soil_nutrients:
            self.soil_nutrients[nutrient] = nutrients_val[nutrient]

# ------------------------------
# 2. Plant Setup
# ------------------------------

class Plant:
    def __init__(self, plant_type: str):
        self.plant_type = plant_type
        self.health = 1.0
        self.biomass = {"leaf": 1.0, "root": 1.0, "stem": 1.0, "storage": 0.5}
        self.leaf_area = self.biomass["leaf"] ** 0.75
        self.glucose_pool = 0.0

    def allocate_growth(self, glucose):
        self.biomass["leaf"] += 0.4 * glucose
        self.biomass["root"] += 0.3 * glucose
        self.biomass["stem"] += 0.2 * glucose
        self.biomass["storage"] += 0.1 * glucose
        self.leaf_area = self.biomass["leaf"] ** 0.75

# ------------------------------
# 3. Photosynthesis Logic
# ------------------------------

class Photosynthesis:
    @staticmethod
    def photosynthesize(plant: Plant, env: Environment):
        # Plant-specific optimal temperatures
        plant_temp_opt = {"C3": 25, "C4": 30, "CAM": 35}
        opt_temp = plant_temp_opt.get(plant.plant_type, 25)
        temp_stress = max(0, 1 - abs(opt_temp - env.temperature) / 40)

        # Nutrient stress
        nutrient_stress = min(env.soil_nutrients.values())

        # Light capture factor
        light_factor = plant.leaf_area * (env.light.value / 4)

        # Total stress
        total_stress = temp_stress * nutrient_stress
        stomatal_opening = total_stress

        # Glucose production
        glucose = light_factor * stomatal_opening * 5
        plant.glucose_pool += glucose

        # Maintenance cost
        maintenance_cost = 0.05 * sum(plant.biomass.values())
        plant.glucose_pool = max(0, plant.glucose_pool - maintenance_cost)

        # Allocate growth
        plant.allocate_growth(plant.glucose_pool)

        # Update health
        plant.health *= total_stress
        plant.health = max(0.0, min(1.0, plant.health))

# ------------------------------
# 4. Simulation Function
# ------------------------------

def run_simulation(light_val=2, temp_val=25, n_days=60,
                   nitrogen=0.8, phosphorus=0.7, potassium=0.6):

    env = Environment(soil_nutrients={"N": nitrogen, "P": phosphorus, "K": potassium},
                      co2_ppm=400, temperature=temp_val)

    wheat = Plant("C3")
    corn = Plant("C4")
    cactus = Plant("CAM")
    plants = [wheat, corn, cactus]

    history = {p.plant_type: {"biomass": [], "health": [], "leaf_area": []} for p in plants}

    nutrients_dict = {"N": nitrogen, "P": phosphorus, "K": potassium}

    for day in range(n_days):
        env.update_conditions(light_val, temp_val, nutrients_dict)
        for plant in plants:
            Photosynthesis.photosynthesize(plant, env)
            history[plant.plant_type]["biomass"].append(sum(plant.biomass.values()))
            history[plant.plant_type]["health"].append(plant.health)
            history[plant.plant_type]["leaf_area"].append(plant.leaf_area)

    # ------------------------------
    # 5. Visualization
    # ------------------------------
    plt.figure(figsize=(12, 4))
    for plant in plants:
        plt.plot(history[plant.plant_type]["biomass"], label=f"{plant.plant_type} Biomass")
    plt.title("Plant Biomass Over Time")
    plt.xlabel("Days")
    plt.ylabel("Biomass")
    plt.legend()
    plt.show()

    plt.figure(figsize=(12, 4))
    for plant in plants:
        plt.plot(history[plant.plant_type]["health"], label=f"{plant.plant_type} Health")
    plt.title("Plant Health Over Time")
    plt.xlabel("Days")
    plt.ylabel("Health")
    plt.legend()
    plt.show()

    plt.figure(figsize=(12, 4))
    for plant in plants:
        plt.plot(history[plant.plant_type]["leaf_area"], label=f"{plant.plant_type} Leaf Area")
    plt.title("Leaf Area Over Time")
    plt.xlabel("Days")
    plt.ylabel("Leaf Area")
    plt.legend()
    plt.show()

# ------------------------------
# 6. Interactive Sliders
# ------------------------------

interact(run_simulation,
         light_val=IntSlider(min=0, max=4, step=1, value=2, description="Light"),
         temp_val=FloatSlider(min=0, max=40, step=0.5, value=25, description="Temperature (°C)"),
         n_days=IntSlider(min=10, max=120, step=10, value=60, description="Simulation Days"),
         nitrogen=FloatSlider(min=0.1, max=1.0, step=0.05, value=0.8, description="Nitrogen"),
         phosphorus=FloatSlider(min=0.1, max=1.0, step=0.05, value=0.7, description="Phosphorus"),
         potassium=FloatSlider(min=0.1, max=1.0, step=0.05, value=0.6, description="Potassium"))

interactive(children=(IntSlider(value=2, description='Light', max=4), FloatSlider(value=25.0, description='Tem…

<function __main__.run_simulation(light_val=2, temp_val=25, n_days=60, nitrogen=0.8, phosphorus=0.7, potassium=0.6)>